# AutoResearch Layer 1: Training Experiments

In [ ]:
# ===== Setup =====
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi 2>/dev/null || echo 'No GPU'

# Install with Drive pip cache
!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache \
    mamba-ssm causal-conv1d einops \
    transformers nibabel pyyaml tqdm scipy

import os, subprocess, zipfile, time, shutil, glob
REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'
DRIVE_CKPT = os.path.join(DRIVE_BASE, 'checkpoints')
os.makedirs(DRIVE_CKPT, exist_ok=True)

# Clone/pull with retry
git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    shutil.rmtree(REPO_DIR)
if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
else:
    for attempt in range(1, 4):
        ret = subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR],
            capture_output=True, text=True)
        if ret.returncode == 0:
            break
        print(f'Clone attempt {attempt} failed: {ret.stderr.strip()}')
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        time.sleep(5 * attempt)
    else:
        raise RuntimeError('Clone failed after 3 attempts')
    os.chdir(REPO_DIR)
print(f'Repo: {os.getcwd()}')

# Data
DATA_DIR = './data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    with zipfile.ZipFile(f'{DRIVE_BASE}/TextBraTS_data.zip', 'r') as zf:
        zf.extractall(os.path.dirname(DATA_DIR))
ET_CACHE = f'{DRIVE_BASE}/et_enriched.zip'
if os.path.exists(ET_CACHE):
    with zipfile.ZipFile(ET_CACHE, 'r') as zf:
        zf.extractall(DATA_DIR)
cases = [d for d in os.listdir(DATA_DIR) if d.startswith('BraTS')]
print(f'Data: {len(cases)} cases')

def sync_and_tag(tag):
    local_ckpt = os.path.join(REPO_DIR, 'checkpoints')
    if not os.path.exists(local_ckpt):
        return
    for f in glob.glob(os.path.join(local_ckpt, '*.pth')):
        shutil.copy2(f, os.path.join(DRIVE_CKPT, os.path.basename(f)))
    best = os.path.join(local_ckpt, 'best.pth')
    if os.path.exists(best):
        shutil.copy2(best, os.path.join(DRIVE_CKPT, f'best_{tag}.pth'))
        print(f'Tagged: best_{tag}.pth')
    print(f'Synced to {DRIVE_CKPT}')

print('Setup complete')

## L1-5: d_state=64 Quick Screen (30 epochs)\n\nQuick test: does Mamba2 with d_state=64 improve over d_state=16?

In [ ]:
# L1-5: d_state=64 quick screen
import os, glob
os.chdir(REPO_DIR)

# Clean checkpoints
for f in glob.glob(os.path.join(REPO_DIR, 'checkpoints', '*.pth')):
    os.remove(f)
print('Cleaned checkpoints')

# Train from scratch, 30 epochs
!python -u train.py \
    --config configs/autoresearch/L1-5_dstate64_quick.yaml \
    --no-text-ratio 0.15 \
    --grad-accum 2

sync_and_tag('L1-5')
print('L1-5 complete')

In [ ]:
# L1-5 Eval
import subprocess, re, os, json
os.chdir(REPO_DIR)

ckpt = os.path.join(DRIVE_CKPT, 'best_L1-5.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, 'checkpoints', 'best.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(DRIVE_CKPT, 'last.pth')
assert os.path.exists(ckpt), f'No checkpoint: {ckpt}'
print(f'Using: {ckpt}')

for name, flags in [('text+TTA', ['--use-text', '--tta']), ('notext+TTA', ['--no-text', '--tta'])]:
    cmd = ['python', '-u', 'evaluate_full.py',
           '--config', 'configs/autoresearch/L1-5_dstate64_quick.yaml',
           '--checkpoint', ckpt, '--split', 'test', '--overlap', '0.5'] + flags
    ret = subprocess.run(cmd, capture_output=True, text=True)
    print(f'--- {name} ---')
    for line in ret.stdout.split(chr(10)):
        if 'dice_' in line or 'hd95_' in line:
            print(line)

## L1-1: Higher LR from Scratch (200 epochs, lr=1e-4)\n\nV5.0 used lr=5e-5. Double it to escape local minimum.

In [ ]:
# L1-1: lr=1e-4 from scratch
import os, glob
os.chdir(REPO_DIR)

for f in glob.glob(os.path.join(REPO_DIR, 'checkpoints', '*.pth')):
    os.remove(f)
print('Cleaned checkpoints')

!python -u train.py \
    --config configs/autoresearch/L1-1_lr1e4_scratch.yaml \
    --no-text-ratio 0.15 \
    --grad-accum 2

sync_and_tag('L1-1')
print('L1-1 complete')

In [ ]:
# L1-1 Eval
import subprocess, os
os.chdir(REPO_DIR)

ckpt = os.path.join(DRIVE_CKPT, 'best_L1-1.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, 'checkpoints', 'best.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(DRIVE_CKPT, 'last.pth')
assert os.path.exists(ckpt), f'No checkpoint: {ckpt}'
print(f'Using: {ckpt}')

for name, flags in [('text+TTA', ['--use-text', '--tta']), ('notext+TTA', ['--no-text', '--tta'])]:
    cmd = ['python', '-u', 'evaluate_full.py',
           '--config', 'configs/autoresearch/L1-1_lr1e4_scratch.yaml',
           '--checkpoint', ckpt, '--split', 'test', '--overlap', '0.5'] + flags
    ret = subprocess.run(cmd, capture_output=True, text=True)
    print(f'--- {name} ---')
    for line in ret.stdout.split(chr(10)):
        if 'dice_' in line or 'hd95_' in line:
            print(line)

## L1-4: ET-Weighted Training (200 epochs)\n\nET class weight 6.0 (vs default 4.0), early stopping on ET Dice.

In [ ]:
# L1-4: ET-weighted
import os, glob
os.chdir(REPO_DIR)

for f in glob.glob(os.path.join(REPO_DIR, 'checkpoints', '*.pth')):
    os.remove(f)

!python -u train.py \
    --config configs/autoresearch/L1-4_et_weighted.yaml \
    --no-text-ratio 0.15 \
    --grad-accum 2 \
    --es-metric et

sync_and_tag('L1-4')
print('L1-4 complete')

In [ ]:
# L1-4 Eval
import subprocess, os
os.chdir(REPO_DIR)

ckpt = os.path.join(DRIVE_CKPT, 'best_L1-4.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, 'checkpoints', 'best.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(DRIVE_CKPT, 'last.pth')
assert os.path.exists(ckpt), f'No checkpoint: {ckpt}'
print(f'Using: {ckpt}')

for name, flags in [('text+TTA', ['--use-text', '--tta']), ('notext+TTA', ['--no-text', '--tta'])]:
    cmd = ['python', '-u', 'evaluate_full.py',
           '--config', 'configs/autoresearch/L1-4_et_weighted.yaml',
           '--checkpoint', ckpt, '--split', 'test', '--overlap', '0.5'] + flags
    ret = subprocess.run(cmd, capture_output=True, text=True)
    print(f'--- {name} ---')
    for line in ret.stdout.split(chr(10)):
        if 'dice_' in line or 'hd95_' in line:
            print(line)

## L1-3: Aggressive Augmentation (200 epochs)\n\nCopy-paste=0.5, no_text_ratio=0.3, all augmentations on.

In [ ]:
# L1-3: Aggressive augmentation
import os, glob
os.chdir(REPO_DIR)

for f in glob.glob(os.path.join(REPO_DIR, 'checkpoints', '*.pth')):
    os.remove(f)

!python -u train.py \
    --config configs/autoresearch/L1-3_aggressive_aug.yaml \
    --grad-accum 2

sync_and_tag('L1-3')
print('L1-3 complete')

In [ ]:
# L1-3 Eval
import subprocess, os
os.chdir(REPO_DIR)

ckpt = os.path.join(DRIVE_CKPT, 'best_L1-3.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, 'checkpoints', 'best.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(DRIVE_CKPT, 'last.pth')
assert os.path.exists(ckpt), f'No checkpoint: {ckpt}'
print(f'Using: {ckpt}')

for name, flags in [('text+TTA', ['--use-text', '--tta']), ('notext+TTA', ['--no-text', '--tta'])]:
    cmd = ['python', '-u', 'evaluate_full.py',
           '--config', 'configs/autoresearch/L1-3_aggressive_aug.yaml',
           '--checkpoint', ckpt, '--split', 'test', '--overlap', '0.5'] + flags
    ret = subprocess.run(cmd, capture_output=True, text=True)
    print(f'--- {name} ---')
    for line in ret.stdout.split(chr(10)):
        if 'dice_' in line or 'hd95_' in line:
            print(line)

## L1 Summary

In [ ]:
# L1 Results Summary
print('AutoResearch Layer 1 Results')
print('=' * 70)
print(f'Baseline V5.0: Mean=0.8479, ET=0.7910, TC=0.8560, WT=0.8967')
print()
print('Check eval cells above for each experiment result.')
print('Record the best result:')
print('  python -m autoresearch record L1-X \'{"dice_mean": 0.XX, "dice_ET": 0.XX}\'')